# Analytics Warehouse Story Notebook

This notebook reads warehouse-ready aggregates and feature-store rows for reproducible DS storytelling.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import pandas as pd
from pymongo import MongoClient

REPO_ROOT = Path.cwd()
ENV_PATH = REPO_ROOT / 'backend' / '.env'

MONGODB_URL = os.getenv('MONGODB_URL', 'mongodb://localhost:27017')
MONGODB_DB_NAME = os.getenv('MONGODB_DB_NAME', 'vidyaverse')

if ENV_PATH.exists():
    for raw in ENV_PATH.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == 'MONGODB_URL' and 'MONGODB_URL' not in os.environ:
            MONGODB_URL = value.strip()
        if key.strip() == 'MONGODB_DB_NAME' and 'MONGODB_DB_NAME' not in os.environ:
            MONGODB_DB_NAME = value.strip()

client = MongoClient(MONGODB_URL)
db = client[MONGODB_DB_NAME]
print('Connected to', MONGODB_DB_NAME)

In [ ]:
LOOKBACK_DAYS = 30
TRAFFIC_TYPE = 'real'

daily = pd.DataFrame(list(db.analytics_daily_aggregates.find({'traffic_type': TRAFFIC_TYPE})))
funnel = pd.DataFrame(list(db.analytics_funnel_aggregates.find({'traffic_type': TRAFFIC_TYPE})))
cohort = pd.DataFrame(list(db.analytics_cohort_aggregates.find({'traffic_type': TRAFFIC_TYPE})))
feature_rows = pd.DataFrame(list(db.feature_store_rows.find({'traffic_type': TRAFFIC_TYPE})))

print('daily rows:', len(daily))
print('funnel rows:', len(funnel))
print('cohort rows:', len(cohort))
print('feature rows:', len(feature_rows))

In [ ]:
if not daily.empty:
    request_daily = daily[daily['metric_type'] == 'request'].copy()
    interaction_daily = daily[daily['metric_type'] == 'interaction'].copy()

    if not interaction_daily.empty:
        interaction_daily = interaction_daily.sort_values('date')
        interaction_daily['ctr'] = interaction_daily['metrics'].apply(lambda x: float((x or {}).get('ctr', 0.0)))
        interaction_daily[['date', 'ranking_mode', 'experiment_variant', 'ctr']].tail(20)


In [ ]:
if not funnel.empty:
    funnel = funnel.sort_values('date')
    funnel['apply_from_click'] = funnel['rates'].apply(lambda x: float((x or {}).get('apply_from_click', 0.0)))
    display(funnel[['date', 'ranking_mode', 'experiment_variant', 'apply_from_click']].tail(20))


In [ ]:
if not cohort.empty:
    cohort = cohort.sort_values(['cohort_date', 'days_since_cohort'])
    recent = cohort[cohort['days_since_cohort'].isin([1, 7, 14, 30])]
    display(recent[['cohort_date', 'days_since_cohort', 'retention_rate', 'apply_rate']].tail(40))


In [ ]:
if not feature_rows.empty:
    feature_rows['label_clicked'] = feature_rows['labels'].apply(lambda x: int((x or {}).get('clicked', 0)))
    feature_rows['label_saved'] = feature_rows['labels'].apply(lambda x: int((x or {}).get('saved', 0)))
    feature_rows['label_applied'] = feature_rows['labels'].apply(lambda x: int((x or {}).get('applied', 0)))
    summary = feature_rows[['label_clicked', 'label_saved', 'label_applied']].mean().rename('rate').to_frame()
    display(summary)


## Export Ready Tables

Use these dataframes (`interaction_daily`, `funnel`, `cohort`, `summary`) to export portfolio artifacts and dashboard screenshots.